In [51]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

import torch.nn as nn
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

In [45]:
# Data loading
df = pd.read_csv('../Data/Training Set filtered.csv')

df.columns = df.columns.str.strip()

df.head

<bound method NDFrame.head of                   Datetime  Export  Import  Generation  National Load  \
0      01/10/2025 05:45:00     0.0  320.38     1157.64        1478.02   
1      01/10/2025 06:00:00     0.0  329.78     1175.98        1505.76   
2      01/10/2025 06:15:00     0.0  377.45     1221.01        1598.46   
3      01/10/2025 06:30:00     0.0  403.71     1231.19        1636.49   
4      01/10/2025 06:45:00     0.0  369.32     1279.29        1648.61   
...                    ...     ...     ...         ...            ...   
37732                  NaN     NaN     NaN         NaN        1284.43   
37733                  NaN     NaN     NaN         NaN        1284.43   
37734                  NaN     NaN     NaN         NaN        1284.43   
37735                  NaN     NaN     NaN         NaN        1284.43   
37736                  NaN     NaN     NaN         NaN        1284.43   

       System Load  Bhairahawa_Temperature (at 2m), C  \
0          1478.02                  

In [46]:
# Small cleanups
df['Datetime'] = pd.to_datetime(df['Datetime'], format = '%m/%d/%Y %H:%M:%S')

# Dropping the NaT trailing rows in Datetime
# df = df.dropna(subset = ['Datetime']).copy()

df = df.dropna()

In [47]:
# Feature engineering
hour_float = df['Datetime'].dt.hour + df['Datetime'].dt.minute / 60
df['sin_hour'] = np.sin(2 * np.pi * hour_float / 24)
df['cos_hour'] = np.cos(2 * np.pi * hour_float / 24)
df['sin_dow'] = np.sin(2 * np.pi * df['Datetime'].dt.dayofweek / 7)
df['cos_dow'] = np.cos(2 * np.pi * df['Datetime'].dt.dayofweek / 7)

In [48]:
# Defining targets

TARGET_COL = "National Load"
FEATURE_COLS = [c for c in df.columns if (c!= 'Datetime' and c!= 'is_outlier')] # using everything
TARGET_IDX = FEATURE_COLS.index(TARGET_COL)
INPUT_DIM = len(FEATURE_COLS)



In [49]:
# Hyperparameters

SEQ_LEN = 96
BATCH_SIZE = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LR = 1e-3
EPOCHS = 50
TRAIN_FRAC = 0.8 # 80 20 split

In [50]:
# Datasplitting

data = df[FEATURE_COLS].values
split = int(len(data) * TRAIN_FRAC)

train_raw, test_raw = data[:split], data[split:]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_raw)
test_scaled = scaler.transform(test_raw)

In [52]:
# Dataloading

class TimeSeriesDataset(Dataset):
    def __init__(self, data: np.ndarray, seq_len: int, target_idx: int):
        self.data = torch.tensor(data, dtype = torch.float32)
        self.seq_len = seq_len
        self.target_idx = target_idx

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len, self.target_idx]
        return x, y


train_ds = TimeSeriesDataset(train_scaled, SEQ_LEN, TARGET_IDX)
test_ds = TimeSeriesDataset(test_scaled, SEQ_LEN, TARGET_IDX)

train_loader = DataLoader(train_ds, batch_size = BATCH_SIZE, shuffle = True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size = BATCH_SIZE, shuffle = True)

xb, yb = next(iter(train_loader))
print(f'Batch X: {xb.shape} (batch, seq_len, features)')
print(f'Batch y: {yb.shape} (batch)')

Batch X: torch.Size([64, 96, 37]) (batch, seq_len, features)
Batch y: torch.Size([64]) (batch)


In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size = input_dim,
            hidden_size = hidden_dim,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout if num_layers > 1 else 0.0,
        )


        # Small MLP head for non-linear mapping from hiddle state -> prediction
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )


    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)


model = LSTMForecaster(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
print(model)

AttributeError: module 'torch.nn' has no attribute 'Lineaer'